# Changes Made

## requirements.txt
- Added dependency:
  - playwright

## pyproject.toml
- Added dependency in `[project].dependencies`:
  - playwright>=1.58.0

## scrap.py
- Created a new scraper module that:
- Uses `requests` + `BeautifulSoup` for standard/static pages.
- Falls back to `playwright` for JavaScript-heavy pages or request failures.
- Cleans page content by removing non-useful tags (`script`, `style`, `img`, `input`).
- Provides `fetch_website_contents(url)` with output truncated to 2,000 characters.
- Provides `fetch_website_links(url)` to extract links from the page.

## Related update
- Notebook import now uses:
  - from scrap import fetch_website_contents


In [1]:
# imports

import os
from dotenv import load_dotenv
from scrap import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 400,000 enrolled across 190

In [4]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [5]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [6]:
openai = OpenAI()
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
response.choices[0].message.content

'2 + 2 equals 4.'

In [7]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [8]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n.

In [9]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [10]:
summarize("https://edwarddonner.com")

"# Edward Donner's AI Playground & Ego Corner\n\nWelcome to Ed’s digital lair where coding meets LLM obsession. Ed is that guy who talks your ear off about large language models (LLMs) until you beg him to write a Udemy course instead—which of course, became a wildly popular smash hit with 400k students pretending to understand what he’s talking about.\n\nIf you’re also into AI, check out his pro courses, battle bots (LLMs duking it out in diplomatic warfare via *Outsmart*), and his ventures (Nebula.io - self-actualization via AI, because why not?). Also, a sprinkle of amateur electronic music production and Hacker News nodding for flavor.\n\n**Recent shiny nuggets:**\n- Feb 17, 2026: AI Coder resources from vibe coder to agentic engineer.\n- Jan 4, 2026: Build AI agents and voice agents with n8n.\n- Sep 15, 2025: Deploy AI models into production MLOps style.\n- May 28, 2025: Guide on the AI courses order — because you don’t want to start with the hard stuff.\n\nGot questions or just w

In [11]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [12]:
display_summary("https://edwarddonner.com")

# Edward Donner's AI Wonderland 🎉

Welcome to the digital lair of Ed, a code junkie and LLM (Large Language Model) whisperer who somehow turned incessant AI rambling into *400,000* happy students worldwide—seriously, who binge-watches Udemy courses like it’s Netflix?

What’s cooking here?  
- An **AI curriculum** so slick it’s broken down into neat courses ranging from *“Vibe Coder to Agentic Engineer”* to *“AI Engineering MLOps Track”* (fancy titles for tinkering with AI in real life).  
- A quirky AI showdown called **Outsmart**, where chatty bots duel in a diplomatic war of wits—basically AI’s version of *Game of Thrones*.  
- Classic geekery: amateur electronic music attempts, and lots of head-nodding at Hacker News articles no one really understands.  

Newsflash: Ed's been dropping fresh AI resources like candy — check out updates from 2025 to early 2026 for cutting-edge AI agent-building and deployment guides.  

If you want to dive into AI without drowning in jargon, or just want to stalk a guy who’s basically the Gandalf of LLMs, this site’s your jam. Plus: newsletter sign-up if you want Ed’s wisdom shoved directly into your inbox.  

Snark level: AI Engineer with a sense of humor—and a playlist of bangers that may or may not exist.

In [13]:
display_summary("https://cnn.com")

# CNN: Your One-Stop Shop for Awesomely Overloaded News and Ads

Welcome to CNN’s digital playground where you can get *all* the news you never asked for, sprinkled with a generous dose of ads that might freeze your screen or blast your eardrums randomly. Want to complain about broken video players or endlessly repeating ads? They've got a feedback form—because your suffering is their content strategy.

Choose your flavor of journalism: US politics, global chaos, celebrity gossip, sports, climate drama, or the latest hot war updates. Oh, and if you’d like, enjoy live TV, multiple editions (because one world version isn't enough), and a dizzying array of newsletters to remind you that there's always *more* news.

TL;DR: CNN is the news buffet of the internet, served with a side of technical glitches and ads that just won’t quit.

In [14]:
display_summary("https://anthropic.com")

# Anthropic: AI with a Conscience

Welcome to Anthropic, the AI overlords—err, researchers—who care deeply about keeping AI *safe* and *useful.* Their mission? Harness AI’s world-shaping power while dodging the usual apocalypse scenarios. They back this up with research, transparency, and fancy-sounding policies like a “Responsible Scaling Policy” and “Claude’s Constitution.” 

Their star product lineup features **Claude**—no, not a French philosopher, but a suite of AI models and platforms including Claude Code and Claude Cowork, boasting the latest “Opus 4.6” model claimed to be *the world’s most powerful* for coding and professional tasks. 

Oh, and they ran a massive study featuring 81,000 people’s opinions on AI (because why not ask when you can?). 

Latest news? As of February 2026, they dropped "Claude Opus 4.6" and reaffirmed their commitment to ad-free, genuinely helpful AI conversations. No nonsense, no spam, just AI that cares about your feelings... or at least about being responsible.

In summary: AI research meets do-gooder vibes with a dash of corporate buzzwords, wrapped in a product suite named after French poets and musical terms.

In [15]:
display_summary("https://openai.com")

# OpenAI Website: TL;DR with Extra Sass

Welcome to the digital lair of OpenAI, where AI magic happens and GPT versions multiply faster than rabbits. The site markets everything from their shiny new GPT-5.4 (because hey, 5.3 had to give way to 5.4 real quick) to the Codex app that probably ghosts your coding nightmares.

**Newsflash:**  
- OpenAI is teaming up with Amazon — because two tech giants are better than one, obviously.  
- There’s some hush-hush government stuff going on with the Department of War (sounds dramatic, but who knows).  
- They’re also obsessing over making math and science learning less painful inside ChatGPT (hallelujah).  
- Codex Security is in “research preview” mode — i.e., playing dress-up in security gear but not quite ready for prime time.

**Research Corner:**  
They’re diving deep into brainy stuff like "Improving instruction hierarchy" and "Reasoning models controlling their chains of thought" (spoiler: it’s messy, but that’s a good thing). Plus, gravity physics sprinkled in with something about gravitons, because why not.

**For Businesses:**  
OpenAI wants to power your taxes, film your social videos, and basically babysit your next-gen talent with ChatGPT.

**In sum:**  
OpenAI’s website is basically a sci-fi buffet: advanced AI updates, corporate drama, some nerdy research, and a sprinkle of 'AI for everyone’ — all served with a side of “Try ChatGPT” button spam.